# Chapter 23 — Imbalance, Thresholds, and Cost-Sensitive Decisions

*From Absolute Zero* — companion notebook.

Every block below is the code printed in the chapter, in the same order. Run the cells top to bottom; the output should match the book exactly. If it does not, check `requirements.txt` first, then `docs/TROUBLESHOOTING.md`.

In [1]:
!pip -q install -r https://raw.githubusercontent.com/FromAbsoluteZero/CodeBase/main/requirements.txt  # Colab only; skip locally
!pip -q install -r https://raw.githubusercontent.com/FromAbsoluteZero/CodeBase/main/requirements-optional.txt  # Colab only; skip locally

zsh:1: command not found: pip


zsh:1: command not found: pip


## Create the data

Run once. Every dataset in this book is generated by code you can read — nothing is downloaded, so nothing can rot behind a dead link. This is the printed block from Chapter 22 (`code/ch22/gen_tx.py` in the repository).

In [2]:
import numpy as np, pandas as pd
rng = np.random.default_rng(21)
n = 60000

amount = np.round(np.exp(rng.normal(3.4, 1.15, n)), 2)     # skewed, as money is
hour = rng.integers(0, 24, n)
age_days = np.clip(rng.gamma(2.0, 260, n), 1, 3000).round(0)
n_country = rng.choice([1, 2, 3], n, p=[0.88, 0.09, 0.03])
prior_chb = rng.poisson(0.06, n)

z = (-9.6
     + 0.95 * np.log1p(amount)
     + 1.60 * ((hour >= 1) & (hour <= 5))
     - 0.0032 * age_days
     + 1.45 * (n_country - 1)
     + 2.10 * prior_chb
     + rng.normal(0, 0.35, n))
fraud = (rng.random(n) < 1 / (1 + np.exp(-z))).astype(int)

pd.DataFrame({"Amount": amount, "Hour": hour, "AccountAgeDays": age_days,
              "CountriesUsed": n_country, "PriorChargebacks": prior_chb,
              "Fraud": fraud}).to_csv("transactions.csv", index=False)
print(f"wrote transactions.csv: {n:,} transactions, "
      f"{fraud.sum():,} fraudulent ({fraud.mean():.3%})")

wrote transactions.csv: 60,000 transactions, 261 fraudulent (0.435%)


## Shared setup

Imports and the objects the blocks below reuse. The chapter prints these once and then continues the same session. This cell is `code/ch23/_lib.py`.

In [3]:
import numpy as np, pandas as pd, warnings; warnings.filterwarnings("ignore")
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (roc_auc_score, average_precision_score,
                             precision_score, recall_score, confusion_matrix)
# transactions.csv is created by Chapter 22 (code/ch22/gen_tx.py). The blocks read it from the
# working directory exactly as the book does; if it is not here yet, use the copy shipped in
# data/generated/ (byte-identical to what the generator writes).
import os as _os, shutil as _shutil
if not _os.path.exists("transactions.csv"):
    for _d in ("../../data/generated", "../data/generated", "data/generated"):
        if _os.path.exists(_os.path.join(_d, "transactions.csv")):
            _shutil.copy(_os.path.join(_d, "transactions.csv"), "transactions.csv"); break
tx = pd.read_csv("transactions.csv")
X = tx.drop(columns="Fraud").values
y = tx["Fraud"].values
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.35,
                                      random_state=0, stratify=y)
C_FP, C_FN = 6.0, 204.0        # review cost; expected loss prevented

## The chapter code

### Block 1  (`c1.py`)

In [4]:
# The break-even threshold falls straight out of the two costs.
p_star = C_FP / (C_FP + C_FN)
print(f"a wasted review costs      {C_FP:>6.0f}")
print(f"a missed fraud costs       {C_FN:>6.0f}")
print(f"break-even threshold p* =  {p_star:.4f}")
print(f"\nflag whenever predicted risk exceeds {p_star:.2%}, not 50%.")
print(f"the default threshold assumes the two errors cost the same,")
print(f"which here would be wrong by a factor of {C_FN/C_FP:.0f}.")

a wasted review costs           6
a missed fraud costs          204
break-even threshold p* =  0.0286

flag whenever predicted risk exceeds 2.86%, not 50%.
the default threshold assumes the two errors cost the same,
which here would be wrong by a factor of 34.


### Block 2  (`c2.py`)

In [5]:
p_star = C_FP / (C_FP + C_FN)

def total_cost(pred):
    fp = int(((pred == 1) & (yte == 0)).sum())
    fn = int(((pred == 0) & (yte == 1)).sum())
    return C_FP * fp + C_FN * fn, fp, fn

print(f"{'model':<10}{'threshold':>11}{'recall':>9}{'precision':>11}"
      f"{'FP':>7}{'FN':>5}{'cost':>10}")
for name, kw in [("plain", {}), ("weighted", {"class_weight": "balanced"})]:
    m = make_pipeline(StandardScaler(),
                      LogisticRegression(max_iter=5000, **kw)).fit(Xtr, ytr)
    pr = m.predict_proba(Xte)[:, 1]
    for th, lab in [(0.5, "0.50 default"), (p_star, f"{p_star:.4f} p*")]:
        pred = (pr >= th).astype(int)
        c, fp, fn = total_cost(pred)
        print(f"{name:<10}{lab:>11}{recall_score(yte, pred):>9.3f}"
              f"{precision_score(yte, pred, zero_division=0):>11.3f}"
              f"{fp:>7}{fn:>5}{c:>10,.0f}")
print(f"\nheld-out set: {len(yte):,} transactions, {yte.sum()} fraudulent")
print(f"cost of flagging nothing at all: {C_FN * yte.sum():>10,.0f}")

model       threshold   recall  precision     FP   FN      cost
plain     0.50 default    0.011      0.250      3   90    18,378
plain       0.0286 p*    0.396      0.069    486   55    14,136
weighted  0.50 default    0.725      0.020   3180   25    24,180
weighted    0.0286 p*    1.000      0.005  19189    0   115,134

held-out set: 21,000 transactions, 91 fraudulent
cost of flagging nothing at all:     18,564


### Block 3  (`c3.py`)

In [6]:
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import make_pipeline as imb_pipeline

cv = StratifiedKFold(5, shuffle=True, random_state=0)

# WRONG: resample everything, then cross-validate. Synthetic minority
# points are built from rows that later serve as validation data.
sm = SMOTE(random_state=0)
Xr, yr = sm.fit_resample(Xtr, ytr)
wrong = cross_val_score(make_pipeline(StandardScaler(),
                        LogisticRegression(max_iter=5000)),
                        Xr, yr, cv=cv, scoring="average_precision").mean()

# RIGHT: resample inside each fold, on that fold's training rows only.
right = cross_val_score(imb_pipeline(StandardScaler(), SMOTE(random_state=0),
                        LogisticRegression(max_iter=5000)),
                        Xtr, ytr, cv=cv, scoring="average_precision").mean()

plain = cross_val_score(make_pipeline(StandardScaler(),
                        LogisticRegression(max_iter=5000)),
                        Xtr, ytr, cv=cv, scoring="average_precision").mean()

print(f"resampled before splitting (WRONG): {wrong:.4f}")
print(f"resampled inside each fold  (right): {right:.4f}")
print(f"no resampling at all               : {plain:.4f}")
print(f"\nthe wrong version is {wrong/right:.0f}x the honest estimate")

resampled before splitting (WRONG): 0.9112
resampled inside each fold  (right): 0.1028
no resampling at all               : 0.1234

the wrong version is 9x the honest estimate


### Block 4  (`c4.py`)

In [7]:
# Does the theoretical p* actually minimize cost on held-out data?
m = make_pipeline(StandardScaler(),
                  LogisticRegression(max_iter=5000)).fit(Xtr, ytr)
pr = m.predict_proba(Xte)[:, 1]
p_star = C_FP / (C_FP + C_FN)

def cost(th):
    pred = pr >= th
    return (C_FP * int((pred & (yte == 0)).sum())
            + C_FN * int((~pred & (yte == 1)).sum()))

grid = np.unique(np.round(np.geomspace(0.0005, 0.5, 400), 6))
costs = np.array([cost(t) for t in grid])
best = grid[costs.argmin()]

print(f"theoretical p*        {p_star:.4f}   cost {cost(p_star):>9,.0f}")
print(f"empirical minimum     {best:.4f}   cost {costs.min():>9,.0f}")
print(f"default 0.50          0.5000   cost {cost(0.5):>9,.0f}")
print(f"flag everything       0.0000   cost {cost(0.0):>9,.0f}")
print(f"flag nothing          1.0000   cost {C_FN*yte.sum():>9,.0f}")
print(f"\nusing p* rather than the empirical optimum costs "
      f"{cost(p_star) - costs.min():,.0f} more")

theoretical p*        0.0286   cost    14,136
empirical minimum     0.0229   cost    13,296
default 0.50          0.5000   cost    18,378
flag everything       0.0000   cost   125,454
flag nothing          1.0000   cost    18,564

using p* rather than the empirical optimum costs 840 more


### Block 5  (`c5.py`)

In [8]:
# Cost is not the only constraint. Capacity usually binds first.
m = make_pipeline(StandardScaler(),
                  LogisticRegression(max_iter=5000)).fit(Xtr, ytr)
pr = m.predict_proba(Xte)[:, 1]
p_star = C_FP / (C_FP + C_FN)

CAPACITY = 200        # reviews this team can actually do
order = np.argsort(-pr)
top = order[:CAPACITY]
flagged_by_pstar = int((pr >= p_star).sum())

print(f"threshold p* would flag       {flagged_by_pstar:,} transactions")
print(f"the team can review           {CAPACITY}")
print(f"implied threshold at capacity {pr[order[CAPACITY-1]]:.4f}")
print()
print(f"reviewing the top {CAPACITY}: caught {yte[top].sum()} of {yte.sum()}"
      f"   precision {yte[top].mean():.1%}")
net = yte[top].sum() * C_FN - CAPACITY * C_FP
print(f"net value {net:,.0f}   versus {C_FN*yte.sum():,.0f} of loss "
      f"if nothing is reviewed")

threshold p* would flag       522 transactions
the team can review           200
implied threshold at capacity 0.0600

reviewing the top 200: caught 23 of 91   precision 11.5%
net value 3,492   versus 18,564 of loss if nothing is reviewed
